# 25 Local Post-to-Trend Matching

Deterministic local-only staged matching from Bluesky topic candidates to normalized Twitter trends.

- No Snowflake
- No reruns of collection/enrichment
- No ML training in this phase


**Notebook purpose:** Runs staged (exact/fuzzy/semantic) matching from Bluesky topic candidates to normalized Twitter trends. Produces full-match and best-match parquet outputs.

**Required data:** `local/derived/bluesky/bluesky_topic_candidates.parquet` (from notebook 24) and `local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet` (from notebook 22).

**Run order:** Run after notebook 24 (topic extraction). Run before notebook 26 (feature engineering).

## 1. Load Inputs and Inspect Schemas


In [7]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'post_trend_matching.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/nlp/post_trend_matching.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.post_trend_matching import MatchConfig, match_post_candidates_to_trends

candidates_path = ROOT / "local/derived/bluesky/bluesky_topic_candidates.parquet"
trends_path = ROOT / "local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet"

for _p, _label in [(candidates_path, "notebook 24 (topic candidates)"), (trends_path, "notebook 22 (trend normalization)")]:
    if not _p.exists():
        raise FileNotFoundError(f"DATA NOT YET AVAILABLE -- run {_label} first.\nMissing: {_p}")

candidates_df = pd.read_parquet(candidates_path)
trends_df = pd.read_parquet(trends_path)

print("Candidate rows:", len(candidates_df), "columns:", len(candidates_df.columns))
print("Trend rows:", len(trends_df), "columns:", len(trends_df.columns))
print("Candidate columns:", candidates_df.columns.tolist())
print("Trend columns:", trends_df.columns.tolist())

Candidate rows: 237351 columns: 17
Trend rows: 101731 columns: 20
Candidate columns: ['uri', 'post_created_at', 'post_text_raw', 'post_text_clean', 'post_text_alnum', 'candidate_phrase_raw', 'candidate_phrase_clean', 'candidate_phrase_alnum', 'candidate_phrase_no_hash', 'candidate_source_type', 'candidate_token_count', 'candidate_char_count', 'candidate_rank_in_post', 'is_hashtag_candidate', 'contains_digit', 'is_unigram_fallback', 'candidate_start_index']
Trend columns: ['num_hours', 'date', 'name', 'counts', 'trend_name_raw', 'trend_name_clean', 'trend_name_clean_no_hash', 'trend_name_clean_no_dollar', 'trend_name_alnum', 'trend_name_token_count', 'trend_name_char_count', 'is_blank_raw', 'is_hashtag', 'has_special_chars', 'has_non_ascii', 'has_url_like', 'normalized_key_with_hash', 'normalized_key_no_hash', 'normalized_key_no_dollar', 'normalized_date']


## 2. Configure and Run Staged Matching


In [8]:
cfg = MatchConfig(
    date_window_days=3,
    fuzzy_min_score=0.88,
    semantic_min_score=0.60,
    max_stage_pool=5000,
    semantic_pool_top_k=250,
)
cfg


MatchConfig(date_window_days=3, fuzzy_min_score=0.88, semantic_min_score=0.6, max_stage_pool=5000, semantic_pool_top_k=250)

In [9]:
full_matches_df, best_matches_df, summary = match_post_candidates_to_trends(
    candidates_df=candidates_df,
    trends_df=trends_df,
    config=cfg,
)

full_matches_df = full_matches_df.sort_values(["candidate_id", "stage_rank", "trend_name_clean"], kind="stable").reset_index(drop=True)
best_matches_df = best_matches_df.sort_values(["uri"], kind="stable").reset_index(drop=True)

print("Full match rows:", len(full_matches_df))
print("Best match rows:", len(best_matches_df))
summary


KeyboardInterrupt: 

## 3. Stage-by-Stage Outcomes


In [ ]:
stage_row_counts = full_matches_df["match_stage"].value_counts(dropna=False)
stage_candidate_counts = (
    full_matches_df.sort_values(["candidate_id", "stage_rank"], kind="stable")
    .groupby("candidate_id", as_index=False)
    .head(1)["match_stage"]
    .value_counts(dropna=False)
)
temporal_pool_counts = (
    full_matches_df.sort_values(["candidate_id", "stage_rank"], kind="stable")
    .groupby("candidate_id", as_index=False)
    .head(1)["temporal_pool_mode"]
    .value_counts(dropna=False)
)

print("Stage row counts:")
print(stage_row_counts)
print()
print("Stage candidate counts:")
print(stage_candidate_counts)
print()
print("Temporal pool counts:")
print(temporal_pool_counts)


## 4. Representative Examples

Includes exact, fuzzy, semantic, ambiguous, and unmatched examples.


In [ ]:
exact_examples = full_matches_df.loc[full_matches_df["match_stage"] == "exact", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "trend_counts", "temporal_pool_mode"
]].head(10)
fuzzy_examples = full_matches_df.loc[full_matches_df["match_stage"] == "fuzzy", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "trend_counts", "temporal_pool_mode"
]].head(10)
semantic_examples = full_matches_df.loc[full_matches_df["match_stage"] == "semantic", [
    "uri", "candidate_phrase_alnum", "trend_name_clean", "match_score", "semantic_token_cosine", "semantic_char_trigram_jaccard", "temporal_pool_mode"
]].head(10)
ambiguous_examples = full_matches_df.loc[full_matches_df["is_ambiguous"], [
    "candidate_id", "uri", "candidate_phrase_alnum", "trend_name_clean", "match_stage", "match_score", "ambiguity_count"
]].head(20)
unmatched_examples = full_matches_df.loc[full_matches_df["match_stage"] == "unmatched", [
    "uri", "candidate_phrase_alnum", "temporal_pool_mode"
]].head(20)

print("Exact examples")
exact_examples


In [ ]:
print("Fuzzy examples")
fuzzy_examples


In [ ]:
print("Semantic examples")
semantic_examples


In [ ]:
print("Ambiguous examples")
ambiguous_examples


In [ ]:
print("Unmatched examples")
unmatched_examples


## 5. Write Required Outputs


In [ ]:
matching_dir = ROOT / "local/derived/matching"
sample_dir = ROOT / "data/samples"
matching_dir.mkdir(parents=True, exist_ok=True)
sample_dir.mkdir(parents=True, exist_ok=True)

full_out = matching_dir / "bluesky_post_trend_matches.parquet"
best_out = matching_dir / "bluesky_post_best_trend_matches.parquet"
sample_parquet_out = sample_dir / "bluesky_post_trend_matches_sample_1000.parquet"
sample_csv_out = sample_dir / "bluesky_post_trend_matches_sample_1000.csv"
summary_out = matching_dir / "bluesky_post_trend_matching_summary.json"

full_matches_df.to_parquet(full_out, index=False)
best_matches_df.to_parquet(best_out, index=False)
full_matches_df.head(1000).to_parquet(sample_parquet_out, index=False)
full_matches_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "25_local_post_to_trend_matching",
    "config": cfg.__dict__,
    "input_paths": {"candidates": str(candidates_path), "trends": str(trends_path)},
    "summary": summary,
    "stage_row_counts": {str(k): int(v) for k, v in stage_row_counts.items()},
    "stage_candidate_counts": {str(k): int(v) for k, v in stage_candidate_counts.items()},
    "temporal_pool_mode_counts": {str(k): int(v) for k, v in temporal_pool_counts.items()},
    "output_paths": {
        "full_matches_parquet": str(full_out),
        "best_matches_parquet": str(best_out),
        "sample_matches_parquet": str(sample_parquet_out),
        "sample_matches_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Wrote:", full_out)
print("Wrote:", best_out)
print("Wrote:", sample_parquet_out)
print("Wrote:", sample_csv_out)
print("Wrote:", summary_out)